In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import random
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def gera_df():

  splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet'}
  df_assin_2_treino = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["train"])
  df_assin_2_teste = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["test"])
  df_assin_2_val = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["validation"])

  df_assin_2 = pd.concat([df_assin_2_treino, df_assin_2_teste, df_assin_2_val])

  df_assin_2['premise_hypothesis'] = df_assin_2['premise'] + ' ' + df_assin_2['hypothesis']

  occurrence_counts = df_assin_2['premise_hypothesis'].value_counts().reset_index()
  occurrence_counts.columns = ['premise_hypothesis', 'occurrence_count']

  df_assin_2 = df_assin_2.merge(occurrence_counts, on='premise_hypothesis', how='left')

  df_assin_2 = df_assin_2.loc[df_assin_2['occurrence_count'] == 1]

  df_assin_2 = df_assin_2[['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score',
        'entailment_judgment']]

  df_assin_2.reset_index(drop=True, inplace=True)

  return df_assin_2

In [4]:
def extract_response_character(response_text):
  """
  Extrai o caractere '0' ou '1' da resposta do modelo.
  Args:
    response_text (str): A string de resposta do modelo.
  Returns:
    str: '0' ou '1' se encontrado, caso contrário, None.
  """
  # Use regex to find '0' or '1' potentially preceded by whitespace at the beginning of the string
  match = re.search(r'^[\s]*([01])', response_text)
  if match:
    return match.group(1)
  return None

print("Function 'extract_response_character' defined.")

Function 'extract_response_character' defined.


In [5]:
def zero_shot_prompt(premise, hypothesis):
  return f"""
   Você é um sistema de Reconhecimento de Inferência Textual (RTE) em Português Brasileiro.

    Tarefa:
    Dada uma PREMISSA e uma HIPÓTESE, responda *apenas* com um único caractere:
    - 0 se a hipótese não é inferida da premissa.
    - 1 se a hipótese é logicamente inferida da premissa.

    Regras obrigatórias:
    - NÃO explique.
    - NÃO acrescente texto.
    - NÃO repita o enunciado.
    - NÃO responda nada além de 0 ou 1.

    Premissa: {premise}
    Hipótese: {hypothesis}

    Resposta:
    """

#Qwen 4B

In [30]:
model_name = "Qwen/Qwen3-4B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [31]:
df_assin_2 = gera_df()

num_records = len(df_assin_2)
random_index = random.randint(0, num_records - 1)
# random_index = 2

premissa = df_assin_2.iloc[random_index]['premise']
hipotese = df_assin_2.iloc[random_index]['hypothesis']
prompt = zero_shot_prompt(premissa, hipotese)


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=10)
prompt_len = inputs["input_ids"].shape[1]
generated_tokens = output[0][prompt_len:]
resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f'Prompt: {prompt}')
print(f'Resposta: {resp}')

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: cd0b469d-bf01-48fc-95cd-f0470b8d84a7)')' thrown while requesting GET https://huggingface.co/datasets/nilc-nlp/assin2/resolve/main/data/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].


Prompt: 
   Você é um sistema de Reconhecimento de Inferência Textual (RTE) em Português Brasileiro.

    Tarefa:
    Dada uma PREMISSA e uma HIPÓTESE, responda *apenas* com um único caractere:
    - 0 se a hipótese não é inferida da premissa.
    - 1 se a hipótese é logicamente inferida da premissa.

    Regras obrigatórias:
    - NÃO explique.
    - NÃO acrescente texto.
    - NÃO repita o enunciado.
    - NÃO responda nada além de 0 ou 1.

    Premissa: Um menino em um morro coberto de neve está vestindo uma blusa vermelha e um chapéu preto e deslizando nos seus joelhos
    Hipótese: Uma criança está escorregando na neve

    Resposta:
    
Resposta:  1
    (O menino é uma


##Teste de consistência

In [32]:
df_assin_2.head()

,sentence_pair_id,premise,hypothesis,relatedness_score,entailment_judgment
0,1,Uma criança risonha está segurando uma pistola...,Uma criança está segurando uma pistola de água,4.5,1
1,2,Os homens estão cuidadosamente colocando as ma...,Os homens estão colocando bagagens dentro do p...,4.5,1
2,3,Uma pessoa tem cabelo loiro e esvoaçante e est...,Um guitarrista tem cabelo loiro e esvoaçante,4.7,1
3,4,Batatas estão sendo fatiadas por um homem,O homem está fatiando a batata,4.7,1
4,5,Um caminhão está descendo rapidamente um morro,Um caminhão está rapidamente descendo o morro,4.9,1


In [33]:
df_assin_2_first_500 = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_first_500)):
    premissa = df_assin_2_first_500.iloc[i]['premise']
    hipotese = df_assin_2_first_500.iloc[i]['hypothesis']

    prompt = zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=10)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'qwen3_4B_{j}'

    df_assin_2_first_500.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))

df_assin_2_first_500.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_qwen3_4B_consistencia.csv')


/tmp/ipython-input-4037526631.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 02:48:53
100 - 2026-01-11 02:49:47
200 - 2026-01-11 02:50:42
300 - 2026-01-11 02:51:36
400 - 2026-01-11 02:52:31


/tmp/ipython-input-4037526631.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))
/tmp/ipython-input-4037526631.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 02:53:25
100 - 2026-01-11 02:54:19
200 - 2026-01-11 02:55:14
300 - 2026-01-11 02:56:08
400 - 2026-01-11 02:57:02


/tmp/ipython-input-4037526631.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))
/tmp/ipython-input-4037526631.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


...

/tmp/ipython-input-4037526631.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))
/tmp/ipython-input-4037526631.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 03:02:31
100 - 2026-01-11 03:03:26
200 - 2026-01-11 03:04:20
300 - 2026-01-11 03:05:15
400 - 2026-01-11 03:06:10


/tmp/ipython-input-4037526631.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))
/tmp/ipython-input-4037526631.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 03:07:05
100 - 2026-01-11 03:08:00
200 - 2026-01-11 03:08:55
300 - 2026-01-11 03:09:49
400 - 2026-01-11 03:10:44


/tmp/ipython-input-4037526631.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))


##Aplicação em todo o dataset


In [34]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  column_name = f'qwen3_4B'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2[column_name]))

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_qwen3_4B.csv')

0 - 2026-01-11 03:11:39
100 - 2026-01-11 03:12:33
200 - 2026-01-11 03:13:28
300 - 2026-01-11 03:14:23
400 - 2026-01-11 03:15:18
500 - 2026-01-11 03:16:12
600 - 2026-01-11 03:17:07
700 - 2026-01-11 03:18:02
800 - 2026-01-11 03:18:57
900 - 2026-01-11 03:19:51
1000 - 2026-01-11 03:20:46
1100 - 2026-01-11 03:21:40
1200 - 2026-01-11 03:22:35
1300 - 2026-01-11 03:23:30
1400 - 2026-01-11 03:24:25
1500 - 2026-01-11 03:25:20
1600 - 2026-01-11 03:26:14
1700 - 2026-01-11 03:27:09
1800 - 2026-01-11 03:28:04
1900 - 2026-01-11 03:28:59
2000 - 2026-01-11 03:29:53
2100 - 2026-01-11 03:30:48
2200 - 2026-01-11 03:31:43
2300 - 2026-01-11 03:32:38
2400 - 2026-01-11 03:33:32
2500 - 2026-01-11 03:34:27
2600 - 2026-01-11 03:35:22
2700 - 2026-01-11 03:36:17
2800 - 2026-01-11 03:37:11
2900 - 2026-01-11 03:38:06
3000 - 2026-01-11 03:39:01
3100 - 2026-01-11 03:39:56
3200 - 2026-01-11 03:40:51
3300 - 2026-01-11 03:41:46
3400 - 2026-01-11 03:42:41
3500 - 2026-01-11 03:43:36
3600 - 2026-01-11 03:44:31
3700 - 2026-0

#Qwen 3 - 4B Instruct

In [6]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

## Teste de consistência

In [8]:
df_assin_2 = gera_df()

In [7]:
df_assin_2_first_500 = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_first_500)):
    premissa = df_assin_2_first_500.iloc[i]['premise']
    hipotese = df_assin_2_first_500.iloc[i]['hypothesis']

    prompt = zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=10)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'qwen3_4B_instruct_{j}'

    df_assin_2_first_500.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))

df_assin_2_first_500.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_qwen3_4B_instruct_consistencia.csv')

KeyboardInterrupt: 

## Aplicação em todo o dataset

In [9]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)


  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2[column_name]))

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_qwen3_4B_instruct.csv')

0 - 2026-01-11 13:33:29
100 - 2026-01-11 13:34:21
200 - 2026-01-11 13:35:14
300 - 2026-01-11 13:36:06
400 - 2026-01-11 13:36:59
500 - 2026-01-11 13:37:52
600 - 2026-01-11 13:38:45
700 - 2026-01-11 13:39:37
800 - 2026-01-11 13:40:30
900 - 2026-01-11 13:41:23
1000 - 2026-01-11 13:42:15
1100 - 2026-01-11 13:43:08
1200 - 2026-01-11 13:44:00
1300 - 2026-01-11 13:44:52
1400 - 2026-01-11 13:45:45
1500 - 2026-01-11 13:46:37
1600 - 2026-01-11 13:47:30
1700 - 2026-01-11 13:48:22
1800 - 2026-01-11 13:49:15
1900 - 2026-01-11 13:50:07
2000 - 2026-01-11 13:51:00
2100 - 2026-01-11 13:51:52
2200 - 2026-01-11 13:52:45
2300 - 2026-01-11 13:53:37
2400 - 2026-01-11 13:54:30
2500 - 2026-01-11 13:55:22
2600 - 2026-01-11 13:56:15
2700 - 2026-01-11 13:57:08
2800 - 2026-01-11 13:58:01
2900 - 2026-01-11 13:58:54
3000 - 2026-01-11 13:59:46
3100 - 2026-01-11 14:00:39
3200 - 2026-01-11 14:01:32
3300 - 2026-01-11 14:02:26
3400 - 2026-01-11 14:03:18
3500 - 2026-01-11 14:04:12
3600 - 2026-01-11 14:05:05
3700 - 2026-0

#Qwen 3 - 4B Thinking                

In [10]:
model_name = "Qwen/Qwen3-4B-Thinking-2507"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

##Teste de consistência

In [11]:
df_assin_2 = gera_df()

df_assin_2_first_500 = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_first_500)):
    premissa = df_assin_2_first_500.iloc[i]['premise']
    hipotese = df_assin_2_first_500.iloc[i]['hypothesis']

    prompt = zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=10)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'qwen3_4B_thinking_{j}'

    df_assin_2_first_500.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))

df_assin_2_first_500.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_qwen3_4B_thinking_consistencia.csv')

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: d293476e-8f78-4ac0-a911-4072f69eff79)')' thrown while requesting GET https://huggingface.co/datasets/nilc-nlp/assin2/resolve/main/data/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].
/tmp/ipython-input-549161630.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 14:58:01
100 - 2026-01-11 14:58:54
200 - 2026-01-11 14:59:48
300 - 2026-01-11 15:00:41
400 - 2026-01-11 15:01:35


/tmp/ipython-input-549161630.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))
/tmp/ipython-input-549161630.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 15:02:29
100 - 2026-01-11 15:03:23
200 - 2026-01-11 15:04:17
300 - 2026-01-11 15:05:11
400 - 2026-01-11 15:06:05


/tmp/ipython-input-549161630.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))
/tmp/ipython-input-549161630.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


...

/tmp/ipython-input-549161630.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))
/tmp/ipython-input-549161630.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 15:11:32
100 - 2026-01-11 15:12:26
200 - 2026-01-11 15:13:21
300 - 2026-01-11 15:14:15
400 - 2026-01-11 15:15:10


/tmp/ipython-input-549161630.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))
/tmp/ipython-input-549161630.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 15:16:04
100 - 2026-01-11 15:16:59
200 - 2026-01-11 15:17:53
300 - 2026-01-11 15:18:48
400 - 2026-01-11 15:19:42


/tmp/ipython-input-549161630.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2_first_500[column_name]))


##Aplicação em todo o dataset

In [ ]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  column_name = f'qwen3_4B_thinking'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: extract_response_character(x), df_assin_2[column_name]))

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_qwen3_4B_thinking.csv')

0 - 2026-01-11 15:20:37
100 - 2026-01-11 15:21:31
200 - 2026-01-11 15:22:25
300 - 2026-01-11 15:23:18
400 - 2026-01-11 15:24:12
500 - 2026-01-11 15:25:05
600 - 2026-01-11 15:25:59
700 - 2026-01-11 15:26:53
800 - 2026-01-11 15:27:47
900 - 2026-01-11 15:28:40
1000 - 2026-01-11 15:29:34
1100 - 2026-01-11 15:30:28
1200 - 2026-01-11 15:31:21
1300 - 2026-01-11 15:32:15
1400 - 2026-01-11 15:33:08
1500 - 2026-01-11 15:34:02
1600 - 2026-01-11 15:34:56
1700 - 2026-01-11 15:35:50
